In [1]:
# This is already installed on Kaggle
# pip install transformers torch datasets scikit-learn numpy accelerate

# If you're on a machine with limited VRAM, also:
!pip install bitsandbytes  # for 4/8-bit quantisation

In [2]:
# Load Hugging Face access token
# Manage your secrets from the "Add-ons" menu in the top navigation of the editor
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HUGGING_FACE_HUB_TOKEN")

In [3]:
'''
Step 1: Load the dataset

The Azaria & Mitchell True-False dataset lives at notrichardren/azaria-mitchell on
HuggingFace. It has ~13.7k statements across 12 topics (cities, companies, animals,
elements, facts, inventions, etc.), each labelled 0 (false) or 1 (true).
'''

from datasets import load_dataset

dataset = load_dataset("notrichardren/azaria-mitchell", split="train")

print(f"Total examples: {len(dataset)}")
print(
    f"Label distribution: {sum(dataset['label'])} true, {len(dataset) - sum(dataset['label'])} false"
)
print('First 5 samples:')
for i in range(5):
    print(dataset[i])

'''
Total examples: 13673
Label distribution: 6795 true, 6878 false
{'claim': 'Ksar El Kebir is a name of a city.', 'label': 1, 'dataset': 'cities', 'qa_type': 0, 'ind': 0}
{'claim': 'The Valley is a name of a country.', 'label': 0, 'dataset': 'capitals', 'qa_type': 0, 'ind': 1}
{'claim': "Water isn't wet.", 'label': 0, 'dataset': 'neg_facts', 'qa_type': 0, 'ind': 2}
{'claim': "The human brain isn't the control center for the body's functions and emotions.", 'label': 0, 'dataset': 'neg_facts', 'qa_type': 0, 'ind': 3}
{'claim': 'Tieli has a population of approximately 390000.', 'label': 1, 'dataset': 'cities', 'qa_type': 0, 'ind': 4}
''';

Total examples: 13673
Label distribution: 6795 true, 6878 false
First 5 samples:
{'claim': 'Ksar El Kebir is a name of a city.', 'label': 1, 'dataset': 'cities', 'qa_type': 0, 'ind': 0}
{'claim': 'The Valley is a name of a country.', 'label': 0, 'dataset': 'capitals', 'qa_type': 0, 'ind': 1}
{'claim': "Water isn't wet.", 'label': 0, 'dataset': 'neg_facts', 'qa_type': 0, 'ind': 2}
{'claim': "The human brain isn't the control center for the body's functions and emotions.", 'label': 0, 'dataset': 'neg_facts', 'qa_type': 0, 'ind': 3}
{'claim': 'Tieli has a population of approximately 390000.', 'label': 1, 'dataset': 'cities', 'qa_type': 0, 'ind': 4}


In [4]:
'''
Step 2: Load the model
'''

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

#"""
# Full precision if you have a 24GB GPU:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
"""

# OR, if VRAM is tight (e.g. 16GB), use 4-bit:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
"""

model.eval()
print(f"Model loaded. Layers: {model.config.num_hidden_layers}")  # 32 for Llama-3-8B

`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded. Layers: 32


In [6]:
'''
Step 3: Extract activations

This is the core mechanics. We register a hook on a specific transformer layer to capture the
hidden state as the model processes each statement. We extract from the last token position
(standard practice — it's where the model "summarises" the whole input).

Which layer to probe? The paper uses middle-to-late layers (typically layers 14–20 for 32-layer
models). We'll try layer 16 as a starting point.

Note on speed: On a single A100 this takes ~5 minutes for 6300 examples. On a consumer 3090/4090,
~15 minutes. If you're CPU-only, limit to max_samples=500 for testing.
'''

import numpy as np
from tqdm import tqdm

PROBE_LAYER = 12  # 0-indexed; Llama-3-8B has 32 layers total

def get_activation(model, tokenizer, statement: str, layer_idx: int, device="cuda"):
    """
    Run one statement through the model and return the hidden state
    at the last token position from the specified layer.
    Returns a 1D numpy array of shape (hidden_size,) = (4096,) for Llama-3-8B.
    """
    # Format as a plain statement — no chat template needed for probing
    inputs = tokenizer(statement, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    activation = {}
    
    def hook_fn(module, input, output):
        # output is (batch, seq_len, hidden_size); grab last token
        activation["hidden"] = output[0, -1, :].detach().cpu().float().numpy()
    
    # Register hook on the desired layer's output
    hook = model.model.layers[layer_idx].register_forward_hook(hook_fn)
    
    with torch.no_grad():
        model(**inputs)
    
    hook.remove()
    return activation["hidden"]


def extract_all_activations(model, tokenizer, dataset, layer_idx, max_samples=None):
    """
    Extract activations for the full dataset (or a subset).
    """
    device = next(model.parameters()).device
    
    activations = []
    labels = []
    
    examples = dataset if max_samples is None else dataset.select(range(max_samples))
    
    for item in tqdm(examples, desc=f"Extracting layer {layer_idx} activations", miniters=10):
        act = get_activation(model, tokenizer, item["claim"], layer_idx, device)
        activations.append(act)
        labels.append(item["label"])
    
    return np.array(activations), np.array(labels)


X, y = extract_all_activations(model, tokenizer, dataset, layer_idx=PROBE_LAYER, max_samples=10_000)
print(f"Activations shape: {X.shape}")
print(f"Labels shape: {y.shape}")

Extracting layer 12 activations: 100%|██████████| 10000/10000 [1:06:23<00:00,  2.51it/s]


Activations shape: (10000, 4096)
Labels shape: (10000,)


In [7]:
'''
Step 4: Train the linear probe
'''

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# Train/test split — stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardise — important for logistic regression on high-dim data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the probe
# max_iter=1000 and C=0.1 (regularisation) are sensible defaults
probe = LogisticRegression(C=0.1, max_iter=1000, solver="lbfgs")
probe.fit(X_train_scaled, y_train)

# Evaluate
y_pred = probe.predict(X_test_scaled)
y_prob = probe.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred, target_names=["False", "True"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

              precision    recall  f1-score   support

       False       0.91      0.89      0.90      1012
        True       0.89      0.91      0.90       988

    accuracy                           0.90      2000
   macro avg       0.90      0.90      0.90      2000
weighted avg       0.90      0.90      0.90      2000

ROC-AUC: 0.9648


In [8]:
'''
Step 5: Test on your own inputs
'''
def probe_statement(statement: str, model, tokenizer, probe, scaler, layer_idx: int):
    """
    Given a statement, return the probe's prediction and confidence.
    """
    device = next(model.parameters()).device
    act = get_activation(model, tokenizer, statement, layer_idx, device)
    act_scaled = scaler.transform(act.reshape(1, -1))
    
    pred = probe.predict(act_scaled)[0]
    prob = probe.predict_proba(act_scaled)[0]
    
    label = "TRUE" if pred == 1 else "FALSE"
    confidence = prob[pred]
    
    print(f"Statement: '{statement}'")
    print(f"Probe says: {label} (confidence: {confidence:.3f})")
    print(f"  P(true)={prob[1]:.3f}, P(false)={prob[0]:.3f}")
    print()

# Test on known true/false statements
test_statements = [
    "The Eiffel Tower is located in Paris.",        # True
    "The Eiffel Tower is located in Berlin.",       # False
    "Water is composed of hydrogen and oxygen.",    # True
    "The sun orbits the Earth.",                    # False
    "Scotland is a country in the United Kingdom.", # True
    "The capital of Australia is Sydney.",          # False (it's Canberra)
]

for s in test_statements:
    probe_statement(s, model, tokenizer, probe, scaler, layer_idx=PROBE_LAYER)

Statement: 'The Eiffel Tower is located in Paris.'
Probe says: TRUE (confidence: 1.000)
  P(true)=1.000, P(false)=0.000

Statement: 'The Eiffel Tower is located in Berlin.'
Probe says: FALSE (confidence: 1.000)
  P(true)=0.000, P(false)=1.000

Statement: 'Water is composed of hydrogen and oxygen.'
Probe says: TRUE (confidence: 1.000)
  P(true)=1.000, P(false)=0.000

Statement: 'The sun orbits the Earth.'
Probe says: FALSE (confidence: 0.994)
  P(true)=0.006, P(false)=0.994

Statement: 'Scotland is a country in the United Kingdom.'
Probe says: TRUE (confidence: 1.000)
  P(true)=1.000, P(false)=0.000

Statement: 'The capital of Australia is Sydney.'
Probe says: FALSE (confidence: 0.951)
  P(true)=0.049, P(false)=0.951



In [9]:
'''
Step 6: Layer sweep (recommended next step)

Different layers encode different things. It's worth sweeping to find the best layer for your probe.
'''
from sklearn.model_selection import cross_val_score

results = {}

for layer_idx in range(0, model.config.num_hidden_layers, 4):  # every 4th layer
    print(f"Testing layer {layer_idx}...")
    X_layer, y_layer = extract_all_activations(model, tokenizer, dataset, layer_idx, max_samples=1000)
    
    X_scaled = StandardScaler().fit_transform(X_layer)
    probe_cv = LogisticRegression(C=0.1, max_iter=500)
    scores = cross_val_score(probe_cv, X_scaled, y_layer, cv=5, scoring="roc_auc")
    results[layer_idx] = scores.mean()
    print(f"  Layer {layer_idx}: AUC = {scores.mean():.4f}")

best_layer = max(results, key=results.get)
print(f"\nBest layer: {best_layer} (AUC={results[best_layer]:.4f})")

Testing layer 0...


Extracting layer 0 activations: 100%|██████████| 1000/1000 [06:39<00:00,  2.51it/s]


  Layer 0: AUC = 0.6113
Testing layer 4...


Extracting layer 4 activations: 100%|██████████| 1000/1000 [06:38<00:00,  2.51it/s]


  Layer 4: AUC = 0.7676
Testing layer 8...


...

  Layer 20: AUC = 0.9196
Testing layer 24...


Extracting layer 24 activations: 100%|██████████| 1000/1000 [06:39<00:00,  2.50it/s]


  Layer 24: AUC = 0.9080
Testing layer 28...


Extracting layer 28 activations: 100%|██████████| 1000/1000 [06:39<00:00,  2.50it/s]


  Layer 28: AUC = 0.9011

Best layer: 12 (AUC=0.9438)


In [10]:
'''
Step 7: Save your probe
'''
import joblib

joblib.dump(probe, "deception_probe.pkl")
joblib.dump(scaler, "deception_scaler.pkl")
np.save("probe_layer.npy", np.array([PROBE_LAYER]))

# Later, reload with:
# probe = joblib.load("deception_probe.pkl")
# scaler = joblib.load("deception_scaler.pkl")